# mmBERT continued pretraining: LoRA на узбекском корпусе

Ноутбук готовит около 10 млн токенов из UzCrawl, FineWeb2 и локального train, удаляет совпадения с NER dev, выполняет Masked Language Modeling через LoRA и сохраняет:

- checkpoints на шагах 400, 800 и 1200;
- финальный LoRA-адаптер;
- объединённый mmBERT checkpoint для последующего NER-обучения;
- loss и perplexity на фиксированном MLM holdout, включая оценку до первого шага.

В Kaggle должны быть подключены те же dataset с проектом и baseline, что и для `03_train_uzbek_ner_kaggle.ipynb`. Для UzCrawl и FineWeb2 включите Internet.

In [ ]:
# Если версии ниже ещё не установлены, выполните клетку и перезапустите session.
# После restart начните выполнение со следующей клетки.
%pip install -q \
    "transformers==4.57.3" \
    "datasets==4.4.1" \
    "accelerate==1.12.0" \
    "peft==0.17.1" \
    "datasketch==1.6.5" \
    "tqdm>=4.66"

In [ ]:
# Конфигурация и те же правила поиска папок, что в 03_train_uzbek_ner_kaggle.ipynb.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Должно выполняться до первого import torch.

from pathlib import Path
from collections import Counter
import json
import math
import random
import re
import sys

MODEL_NAME = "jhu-clsp/mmBERT-base"
MODEL_REVISION = "c5955035435e2bf121cde7f3c8863ef52ff35d82"
UZCRAWL_REVISION = "8fff2d17bb2607c5d875199c8a7256a4a7ce7e39"
FINEWEB2_REVISION = "af9c13333eb981300149d5ca60a8e9d659b276b9"

SEED = 42
MAX_LENGTH = 256
CONTENT_LENGTH = 254
MAX_DOC_TOKENS = 2_048
MAX_DOC_CHARS = 30_000
MIN_EXTERNAL_CHARS = 50
MIN_EXTERNAL_LETTERS = 20
CYRILLIC_DOMINANCE = 0.80
NEAR_DUPLICATE_THRESHOLD = 0.80
MINHASH_PERMUTATIONS = 128
STREAM_SHUFFLE_BUFFER = 10_000

MAX_STEPS = 1_200
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MLM_PROBABILITY = 0.15
EVAL_AND_SAVE_STEPS = 400

# 38 400 train-блоков: 40% Telegram, 40% news, 10% FineWeb2, 10% local train.
# В сумме 40% блоков кириллические и 60% — латинские/смешанные/иностранные.
TRAIN_QUOTAS = {
    "uzcrawl_telegram": {"cyrillic": 10_080, "other": 5_280},
    "uzcrawl_news": {"cyrillic": 384, "other": 14_976},
    "fineweb2_cyrillic": {"cyrillic": 3_840},
    "local_train": {"cyrillic": 1_056, "other": 2_784},
}

# 640 disjoint holdout-блоков с теми же source/script пропорциями.
HOLDOUT_QUOTAS = {
    "uzcrawl_telegram": {"cyrillic": 168, "other": 88},
    "uzcrawl_news": {"cyrillic": 6, "other": 250},
    "fineweb2_cyrillic": {"cyrillic": 64},
    "local_train": {"cyrillic": 18, "other": 46},
}

SOURCE_ORDER = (
    "local_train",
    "uzcrawl_telegram",
    "uzcrawl_news",
    "fineweb2_cyrillic",
)

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/mmbert_cpt_lora")
PREPARED_DATA_DIR = OUTPUT_ROOT / "prepared_blocks"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
ADAPTER_DIR = OUTPUT_ROOT / "final_adapter"
MERGED_MODEL_DIR = OUTPUT_ROOT / "merged_model"
METRICS_PATH = OUTPUT_ROOT / "holdout_metrics.json"
RUN_SUMMARY_PATH = OUTPUT_ROOT / "run_summary.json"

common_matches = list(INPUT_ROOT.rglob("baseline/common.py"))
if not common_matches:
    raise FileNotFoundError("Не найден baseline/common.py внутри /kaggle/input")
COMMON_PATH = common_matches[0]
PROJECT_DIR = COMMON_PATH.parent.parent

data_dirs = [
    path.parent
    for path in INPUT_ROOT.rglob("train.jsonl")
    if (path.parent / "dev.jsonl").exists()
]
if not data_dirs:
    raise FileNotFoundError("Не найдена папка с train.jsonl и dev.jsonl")
DATA_DIR = data_dirs[0]

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:      ", PROJECT_DIR)
print("DATA_DIR:         ", DATA_DIR)
print("OUTPUT_ROOT:      ", OUTPUT_ROOT)

In [ ]:
# Окружение, tokenizer и локальные normalized train/dev.
import html
import unicodedata

import datasets
import peft
import torch
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import transformers
from datasets import Dataset, DatasetDict, load_dataset, load_from_disk
from datasketch import MinHash, MinHashLSH
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModelForMaskedLM, AutoTokenizer, set_seed

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from baseline.common import read_records

# Та же length-preserving нормализация апострофов, что применена к train.jsonl/dev.jsonl.
CANONICAL_APOSTROPHE = "ʻ"
APOSTROPHE_VARIANTS = {"'", "’", "‘", "ʼ", "`", "´", "ʹ", "ʾ", "ʿ", "＇"}
ENGLISH_APOSTROPHE_ENDINGS = {"s", "t", "re", "ve", "ll", "d", "m"}


def normalize_text(text, *, preserve_english):
    chars = list(text)
    changes = Counter()
    for index, character in enumerate(chars):
        if character not in APOSTROPHE_VARIANTS or character == CANONICAL_APOSTROPHE:
            continue
        if index == 0 or index + 1 == len(chars):
            continue
        if not (chars[index - 1].isalnum() and chars[index + 1].isalnum()):
            continue
        if preserve_english:
            left = index
            while left > 0 and (text[left - 1].isalnum() or text[left - 1] in APOSTROPHE_VARIANTS):
                left -= 1
            right = index + 1
            while right < len(text) and (text[right].isalnum() or text[right] in APOSTROPHE_VARIANTS):
                right += 1
            before, after = text[left:index], text[index + 1:right]
            english_ending = after.casefold() in ENGLISH_APOSTROPHE_ENDINGS and before.isascii() and after.isascii()
            english_name = before.casefold() in {"o", "d"} and after[:1].isupper() and after.isascii()
            if english_ending or english_name:
                continue
        chars[index] = CANONICAL_APOSTROPHE
        changes[character] += 1
    return "".join(chars), changes

if not torch.cuda.is_available():
    raise RuntimeError("Включите GPU в Kaggle: Settings → Accelerator")
if torch.cuda.device_count() != 1:
    raise RuntimeError("Ожидалась одна видимая GPU. Перезапустите session и выполните клетки с начала.")

set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    use_fast=True,
)

if not tokenizer.is_fast:
    raise ValueError("Нужен fast tokenizer mmBERT")
if tokenizer.num_special_tokens_to_add(pair=False) != 2:
    raise ValueError("Ожидались два служебных токена <bos>/<eos>")
if CONTENT_LENGTH + 2 != MAX_LENGTH:
    raise ValueError("CONTENT_LENGTH должен оставлять место для <bos>/<eos>")

local_train_records = read_records(DATA_DIR / "train.jsonl", require_entities=True)
dev_records = read_records(DATA_DIR / "dev.jsonl", require_entities=True)

print("GPU:         ", torch.cuda.get_device_name(0))
print("torch:       ", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:    ", datasets.__version__)
print("peft:        ", peft.__version__)
print("Tokenizer:   ", tokenizer.vocab_size, tokenizer.bos_token, tokenizer.eos_token)
print("Local train: ", len(local_train_records))
print("NER dev:     ", len(dev_records))

In [ ]:
# Очистка, определение письменности, near-deduplication и упаковка токенов.
SCRIPT_STYLE_RE = re.compile(r"(?is)<(script|style)\b.*?>.*?</\1>")
HTML_TAG_RE = re.compile(r"(?s)<[^>]{1,200}>")
WORD_RE = re.compile(r"\w+", flags=re.UNICODE)
WHITESPACE_RE = re.compile(r"\s+")
ZERO_WIDTH = dict.fromkeys(map(ord, "\u200b\u200c\u200d\ufeff"), None)
SPECIAL_IDS = set(tokenizer.all_special_ids)


def clean_external_text(raw_text):
    text = html.unescape(str(raw_text)).translate(ZERO_WIDTH)
    text = SCRIPT_STYLE_RE.sub(" ", text)
    if "<" in text and ">" in text:
        text = HTML_TAG_RE.sub(" ", text)
    text = WHITESPACE_RE.sub(" ", text).strip()[:MAX_DOC_CHARS]
    text, _ = normalize_text(text, preserve_english=True)
    letter_count = sum(character.isalpha() for character in text)
    if len(text) < MIN_EXTERNAL_CHARS or letter_count < MIN_EXTERNAL_LETTERS:
        return None
    return text


def prepare_local_text(raw_text):
    text, _ = normalize_text(str(raw_text), preserve_english=True)
    return text if text.strip() else None


def script_group(text):
    latin = 0
    cyrillic = 0
    for character in text:
        if not character.isalpha():
            continue
        unicode_name = unicodedata.name(character, "")
        latin += "LATIN" in unicode_name
        cyrillic += "CYRILLIC" in unicode_name
    total = latin + cyrillic
    if total and cyrillic / total >= CYRILLIC_DOMINANCE:
        return "cyrillic"
    return "other"


def dedupe_key(text):
    normalized, _ = normalize_text(text, preserve_english=False)
    return WHITESPACE_RE.sub(" ", normalized.casefold()).strip()


def make_minhash(key):
    words = WORD_RE.findall(key)
    width = min(5, len(words))
    shingles = (
        [" ".join(words[index : index + width]) for index in range(len(words) - width + 1)]
        if width
        else [key]
    )
    signature = MinHash(num_perm=MINHASH_PERMUTATIONS, seed=SEED)
    for shingle in shingles:
        signature.update(shingle.encode("utf-8"))
    return signature


class DedupeIndex:
    def __init__(self, reference_texts):
        self.exact = set()
        self.lsh = MinHashLSH(
            threshold=NEAR_DUPLICATE_THRESHOLD,
            num_perm=MINHASH_PERMUTATIONS,
        )
        self.next_id = 0
        for text in tqdm(reference_texts, desc="Index NER dev", unit="doc"):
            self.reject_or_add(text, "dev")

    def reject_or_add(self, text, prefix):
        key = dedupe_key(text)
        if key in self.exact:
            return True
        signature = make_minhash(key)
        if self.lsh.query(signature):
            return True
        item_id = f"{prefix}:{self.next_id}"
        self.next_id += 1
        self.exact.add(key)
        self.lsh.insert(item_id, signature)
        return False


def make_block(content_ids):
    input_ids = [tokenizer.bos_token_id, *content_ids, tokenizer.eos_token_id]
    if len(input_ids) != MAX_LENGTH:
        raise AssertionError("Неверная длина MLM-блока")
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * MAX_LENGTH,
        "special_tokens_mask": [int(token_id in SPECIAL_IDS) for token_id in input_ids],
    }


def fill_blocks(rows, source_name, phase, quotas, *, external, deduper, stats):
    remaining = dict(quotas)
    buffers = {group: [] for group in quotas}
    blocks = []

    for row in rows:
        if not any(remaining.values()):
            break
        stats[f"{source_name}.{phase}.seen_docs"] += 1
        raw_text = row.get("text", "")
        text = clean_external_text(raw_text) if external else prepare_local_text(raw_text)
        if text is None:
            stats[f"{source_name}.{phase}.filtered_docs"] += 1
            continue

        group = script_group(text)
        if remaining.get(group, 0) == 0:
            continue
        if deduper.reject_or_add(text, source_name):
            stats[f"{source_name}.{phase}.duplicate_docs"] += 1
            continue

        token_ids = tokenizer(
            text,
            add_special_tokens=False,
            truncation=False,
            verbose=False,
        )["input_ids"][:MAX_DOC_TOKENS]
        if not token_ids:
            continue

        stats[f"{source_name}.{phase}.accepted_docs"] += 1
        buffer = buffers[group]
        buffer.extend(int(token_id) for token_id in token_ids)
        buffer.append(tokenizer.eos_token_id)

        while remaining[group] and len(buffer) >= CONTENT_LENGTH:
            content_ids = buffer[:CONTENT_LENGTH]
            del buffer[:CONTENT_LENGTH]
            blocks.append(make_block(content_ids))
            remaining[group] -= 1
            stats[f"{source_name}.{phase}.{group}_blocks"] += 1

    if any(remaining.values()):
        raise RuntimeError(f"{source_name}/{phase}: не набраны квоты {remaining}")
    return blocks

In [ ]:
# Потоковые источники и подготовка 38 400 train + 640 holdout блоков.
def local_rows(seed):
    order = list(range(len(local_train_records)))
    random.Random(seed).shuffle(order)
    for index in order:
        yield local_train_records[index]


def source_rows(source_name):
    if source_name == "local_train":
        return iter(local_rows(SEED + 1))
    if source_name == "uzcrawl_telegram":
        dataset = load_dataset(
            "tahrirchi/uz-crawl",
            split="telegram_blogs",
            streaming=True,
            revision=UZCRAWL_REVISION,
        )
        return iter(dataset.shuffle(seed=SEED + 2, buffer_size=STREAM_SHUFFLE_BUFFER))
    if source_name == "uzcrawl_news":
        dataset = load_dataset(
            "tahrirchi/uz-crawl",
            split="news",
            streaming=True,
            revision=UZCRAWL_REVISION,
        )
        return iter(dataset.shuffle(seed=SEED + 3, buffer_size=STREAM_SHUFFLE_BUFFER))
    if source_name == "fineweb2_cyrillic":
        dataset = load_dataset(
            "HuggingFaceFW/fineweb-2",
            "uzn_Cyrl",
            split="train",
            streaming=True,
            revision=FINEWEB2_REVISION,
        )
        return iter(dataset.shuffle(seed=SEED + 4, buffer_size=STREAM_SHUFFLE_BUFFER))
    raise ValueError(f"Неизвестный источник: {source_name}")


expected_train_blocks = sum(sum(groups.values()) for groups in TRAIN_QUOTAS.values())
expected_holdout_blocks = sum(sum(groups.values()) for groups in HOLDOUT_QUOTAS.values())
assert expected_train_blocks == 38_400
assert expected_holdout_blocks == 640

if PREPARED_DATA_DIR.exists():
    prepared = load_from_disk(PREPARED_DATA_DIR)
    print("Загружены ранее подготовленные блоки:", PREPARED_DATA_DIR)
else:
    deduper = DedupeIndex(record["text"] for record in dev_records)
    stats = Counter()
    train_blocks = []
    holdout_blocks = []

    for source_name in SOURCE_ORDER:
        print(f"\nПодготовка {source_name}")
        rows = source_rows(source_name)
        external = source_name != "local_train"
        holdout_blocks.extend(
            fill_blocks(
                rows,
                source_name,
                "holdout",
                HOLDOUT_QUOTAS[source_name],
                external=external,
                deduper=deduper,
                stats=stats,
            )
        )
        train_blocks.extend(
            fill_blocks(
                rows,
                source_name,
                "train",
                TRAIN_QUOTAS[source_name],
                external=external,
                deduper=deduper,
                stats=stats,
            )
        )

    random.Random(SEED).shuffle(train_blocks)
    random.Random(SEED + 10).shuffle(holdout_blocks)
    prepared = DatasetDict(
        {
            "train": Dataset.from_list(train_blocks),
            "validation": Dataset.from_list(holdout_blocks),
        }
    )
    prepared.save_to_disk(PREPARED_DATA_DIR)
    corpus_stats = {
        "train_blocks": len(train_blocks),
        "holdout_blocks": len(holdout_blocks),
        "train_content_tokens": len(train_blocks) * CONTENT_LENGTH,
        "holdout_content_tokens": len(holdout_blocks) * CONTENT_LENGTH,
        "counters": dict(sorted(stats.items())),
    }
    (OUTPUT_ROOT / "corpus_stats.json").write_text(
        json.dumps(corpus_stats, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

if len(prepared["train"]) != expected_train_blocks:
    raise ValueError("Неверное число train-блоков")
if len(prepared["validation"]) != expected_holdout_blocks:
    raise ValueError("Неверное число holdout-блоков")
if any(len(row["input_ids"]) != MAX_LENGTH for row in prepared["train"].select(range(10))):
    raise ValueError("Неверная длина подготовленного блока")

print("Train blocks:   ", len(prepared["train"]))
print("Holdout blocks: ", len(prepared["validation"]))
print("Train tokens:   ", len(prepared["train"]) * CONTENT_LENGTH)

In [ ]:
# В train маски создаются динамически; holdout получает один фиксированный mask pattern.
from transformers import DataCollatorForLanguageModeling, default_data_collator


def fixed_holdout_mask(example, index):
    generator = torch.Generator().manual_seed(SEED + 100_000 + index)
    input_ids = torch.tensor(example["input_ids"], dtype=torch.long)
    labels = input_ids.clone()
    special_mask = torch.tensor(example["special_tokens_mask"], dtype=torch.bool)
    candidates = ~special_mask

    masked = (torch.rand(MAX_LENGTH, generator=generator) < MLM_PROBABILITY) & candidates
    if not masked.any():
        candidate_indices = candidates.nonzero(as_tuple=True)[0]
        masked[candidate_indices[index % len(candidate_indices)]] = True
    labels[~masked] = -100

    replaced = (torch.rand(MAX_LENGTH, generator=generator) < 0.80) & masked
    input_ids[replaced] = tokenizer.mask_token_id

    random_replacements = (
        (torch.rand(MAX_LENGTH, generator=generator) < 0.50)
        & masked
        & ~replaced
    )
    random_tokens = torch.randint(
        low=0,
        high=tokenizer.vocab_size,
        size=(MAX_LENGTH,),
        generator=generator,
    )
    input_ids[random_replacements] = random_tokens[random_replacements]

    return {
        "input_ids": input_ids.tolist(),
        "attention_mask": example["attention_mask"],
        "labels": labels.tolist(),
    }


train_dataset = prepared["train"]
holdout_dataset = prepared["validation"].map(
    fixed_holdout_mask,
    with_indices=True,
    remove_columns=prepared["validation"].column_names,
    desc="Fix holdout masks",
)
train_mlm_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROBABILITY,
)


class TrainAndFixedEvalCollator:
    def __call__(self, features):
        if "labels" in features[0]:
            return default_data_collator(features)
        return train_mlm_collator(features)


data_collator = TrainAndFixedEvalCollator()
masked_holdout_tokens = sum(
    label != -100
    for row in holdout_dataset
    for label in row["labels"]
)
print("Fixed masked holdout tokens:", masked_holdout_tokens)

In [ ]:
# mmBERT Masked Language Model + LoRA.
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
)
config.sparse_prediction = True
config.reference_compile = False  # torch.compile несовместим с DataParallel в этом окружении.

base_model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    config=config,
    attn_implementation="sdpa",
)
if not getattr(base_model, "sparse_prediction", False):
    raise RuntimeError("sparse_prediction не включён до создания модели")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["Wqkv", "Wi", "Wo"],
    bias="none",
)
model = get_peft_model(base_model, lora_config)

matched_modules = [
    name
    for name, module in model.named_modules()
    if hasattr(module, "lora_A") and len(module.lora_A) > 0
]
trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)

if len(matched_modules) != 88:
    raise RuntimeError(f"Ожидалось 88 LoRA-модулей, найдено {len(matched_modules)}")
if trainable_parameters != 3_379_200:
    raise RuntimeError(
        f"Ожидалось 3 379 200 обучаемых параметров, найдено {trainable_parameters:,}"
    )

model.print_trainable_parameters()
print("Matched LoRA modules:", len(matched_modules))

In [ ]:
# Trainer и запись holdout loss/perplexity после каждой оценки.
from transformers import Trainer, TrainerCallback, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint


class HoldoutMetricsCallback(TrainerCallback):
    def __init__(self, output_path):
        self.output_path = Path(output_path)
        self.history = []
        if self.output_path.exists():
            self.history = json.loads(self.output_path.read_text(encoding="utf-8"))

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        metrics = metrics or {}
        loss_key = next((key for key in metrics if key.endswith("_loss")), None)
        if loss_key is None:
            return
        loss = float(metrics[loss_key])
        prefix = loss_key[: -len("_loss")]
        row = {
            "phase": prefix,
            "step": int(state.global_step),
            "loss": loss,
            "perplexity": math.exp(min(loss, 50.0)),
        }
        self.history.append(row)
        self.output_path.write_text(
            json.dumps(self.history, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )
        print(
            f"Holdout {prefix}: step={row['step']}, "
            f"loss={row['loss']:.6f}, perplexity={row['perplexity']:.2f}"
        )


metrics_callback = HoldoutMetricsCallback(METRICS_PATH)
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    max_steps=MAX_STEPS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,
    optim="adamw_torch",
    fp16=True,
    gradient_checkpointing=False,
    eval_strategy="steps",
    eval_steps=EVAL_AND_SAVE_STEPS,
    save_strategy="steps",
    save_steps=EVAL_AND_SAVE_STEPS,
    save_total_limit=3,
    logging_strategy="steps",
    logging_steps=20,
    prediction_loss_only=True,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=holdout_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[metrics_callback],
)

print("Effective batch size:", TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Optimizer steps:      ", MAX_STEPS)
print("Checkpoints:          ", [400, 800, 1200])

In [ ]:
# Валидация исходного mmBERT, обучение/возобновление и финальная валидация.
last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))

if last_checkpoint is None:
    print("Holdout validation before training")
    before_metrics = trainer.evaluate(metric_key_prefix="before")
else:
    print("Resume from:", last_checkpoint)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
final_metrics = trainer.evaluate(metric_key_prefix="final")
peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3

trainer.save_metrics("train", train_result.metrics)
trainer.save_metrics("final", final_metrics)
trainer.save_state()

print(f"Peak allocated GPU memory: {peak_memory_gb:.2f} GiB")
print("Final holdout loss:", final_metrics["final_loss"])

In [ ]:
# Финальный LoRA-адаптер и объединённый checkpoint для AutoModelForTokenClassification.
peft_model = trainer.accelerator.unwrap_model(trainer.model)
peft_model.save_pretrained(ADAPTER_DIR, safe_serialization=True)
tokenizer.save_pretrained(ADAPTER_DIR)

merged_model = peft_model.merge_and_unload()
merged_model.save_pretrained(MERGED_MODEL_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_MODEL_DIR)

run_summary = {
    "base_model": MODEL_NAME,
    "base_model_revision": MODEL_REVISION,
    "uzcrawl_revision": UZCRAWL_REVISION,
    "fineweb2_revision": FINEWEB2_REVISION,
    "seed": SEED,
    "train_blocks": len(train_dataset),
    "holdout_blocks": len(holdout_dataset),
    "content_tokens": len(train_dataset) * CONTENT_LENGTH,
    "max_steps": MAX_STEPS,
    "effective_batch_size": TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "lora_trainable_parameters": trainable_parameters,
    "peak_allocated_gpu_memory_gib": peak_memory_gb,
    "final_holdout_loss": float(final_metrics["final_loss"]),
    "holdout_history": metrics_callback.history,
}
RUN_SUMMARY_PATH.write_text(
    json.dumps(run_summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("LoRA adapter: ", ADAPTER_DIR)
print("Merged model: ", MERGED_MODEL_DIR)
print("Metrics:      ", METRICS_PATH)
print("Run summary:  ", RUN_SUMMARY_PATH)